# E-commerce CSV → MongoDB loader (data_load)

Učitava e-commerce dataset (4 CSV fajla, delimiter `;`) u MongoDB u **dve verzije** šeme,
za projekat iz optimizacije upita:

* **Faza 1 — baseline (`{db}_v1`)**: namerno neoptimalno. 4 odvojene kolekcije
  (`persons`, `products`, `orders`, `orderlines`), sve vrednosti kao **stringovi**,
  ravna polja, podrazumevani `_id` (ObjectId), **bez indeksa**.
* **Faza 2 — optimizovano (`{db}_v2`)**: 3 kolekcije (`persons`, `products`, `orders`),
  orderline **ugnježden** u `orders.lines[]`, prirodni ključ kao `_id`, kastovanje tipova
  (int/float/datetime), poddokumenti, i ciljani **indeksi** (kreirani posle učitavanja).

Sve se čita **streaming-om** (bez učitavanja celog fajla u memoriju). `orderline.csv`
nije sortiran po `order_id`, pa ga Faza 2 prvo **eksterno sortira** na disku
(k-way merge sortiranih chunk-ova, ograničena memorija), zatim radi streaming merge sa
`order.csv`.

**Pokretanje u notebook-u**: izmeni `NB_*` konstante u ćeliji *Konfiguracija* i pokreni sve ćelije.
**Pokretanje kao skripta**: `jupyter nbconvert --to script data_load.ipynb` pa
`python data_load.py --data-dir archive --drop` (vidi README).

In [11]:
"""Streaming CSV → MongoDB loader sa dve verzije šeme (baseline v1 + optimized v2).

Modul je dizajniran da radi i kao Jupyter notebook (koristi NB_* konstante) i kao
samostalna skripta (argparse CLI). Bez učitavanja celog dataseta u memoriju.
"""
from __future__ import annotations

import argparse
import csv
import heapq
import os
import sys
import tempfile
from datetime import datetime
from typing import Any, Callable, Iterable, Iterator, Optional

from pymongo import ASCENDING, DESCENDING, MongoClient
from pymongo.collection import Collection
from pymongo.database import Database

try:
    from tqdm import tqdm
except ImportError:  # tqdm je opcioni; fallback ispisuje progres na ~50k redova
    tqdm = None  # type: ignore[assignment]

# Veliki tekstualni opisi mogu preći podrazumevani limit csv polja.
try:
    csv.field_size_limit(sys.maxsize)
except OverflowError:
    csv.field_size_limit(2 ** 31 - 1)

## Konstante
Imena fajlova i podešavanja. Izmeni `*_CSV` ako se fajlovi drugačije zovu u `--data-dir`.

In [12]:
# --- Imena CSV fajlova (override ovde po potrebi) ---
PERSON_CSV = "person.csv"
PRODUCT_CSV = "product.csv"
ORDER_CSV = "order.csv"
ORDERLINE_CSV = "orderline.csv"

DELIMITER = ";"
NULL_TOKENS = {"", "NULL"}  # prazan string ili "NULL" -> None

# --- Default-i baze/batch-ovanja (takođe izloženi preko CLI) ---
DEFAULT_URI = "mongodb://localhost:27017"
DEFAULT_DB = "ecommerce"
DEFAULT_BATCH_SIZE = 10_000

# Veličina chunk-a (broj redova u memoriji) za eksterno sortiranje orderline.csv.
# Manje = manje RAM-a, više temp fajlova. Podesi prema dostupnoj memoriji.
SORT_CHUNK_ROWS = 500_000

# --- Mape za kastovanje tipova (Faza 2) ---
INT_FIELDS = {"age", "quantity", "stock_quantity", "review_count", "orderline_id"}
FLOAT_FIELDS = {
    "price", "cost", "total_amount", "unit_price", "subtotal",
    "weight_kg", "length_cm", "width_cm", "height_cm", "rating_average",
}
DATETIME_FIELDS = {"order_date", "created_date"}
DATETIME_FORMATS = ("%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d")

## Konfiguracija (CLI + notebook fallback)

In [13]:
def build_parser() -> argparse.ArgumentParser:
    """argparse CLI za pokretanje kao skripta."""
    p = argparse.ArgumentParser(
        description="Učitaj e-commerce CSV-ove u MongoDB (v1 baseline + v2 optimized).",
    )
    p.add_argument("--uri", default=DEFAULT_URI, help="MongoDB URI")
    p.add_argument("--db", default=DEFAULT_DB, help="Bazni naziv -> {db}_v1 i {db}_v2")
    p.add_argument("--data-dir", required=True, help="Direktorijum sa CSV fajlovima")
    p.add_argument("--phase", choices=["1", "2", "both"], default="both")
    p.add_argument("--limit", type=int, default=None,
                   help="Maks. broj porudžbina (prvih N order_id-eva); None = sve")
    p.add_argument("--max-orderlines", type=int, default=None,
                   help="Maks. ukupan broj stavki (zadrži najniže order_id-eve); None = sve")
    p.add_argument("--batch-size", type=int, default=DEFAULT_BATCH_SIZE)
    p.add_argument("--drop", action="store_true",
                   help="Prvo obriši ciljne kolekcije (idempotentno)")
    return p


def _in_notebook() -> bool:
    """True kada se izvršava u Jupyter kernelu."""
    return "ipykernel" in sys.modules or "ipykernel_launcher" in (sys.argv[0] if sys.argv else "")


# --- Notebook default-i (izmeni pri interaktivnom radu) ---
NB_DATA_DIR = "archive"
NB_URI = DEFAULT_URI
NB_DB = DEFAULT_DB
NB_PHASE = "both"
NB_LIMIT: Optional[int] = 1_000_000           # maks. broj porudžbina (orders); None = sve
NB_MAX_ORDERLINES: Optional[int] = 1_000_000  # maks. broj stavki (orderlines); None = sve
NB_BATCH_SIZE = DEFAULT_BATCH_SIZE
NB_DROP = True


class Config:
    """Parametri pokretanja, jedinstveni za notebook i CLI."""

    def __init__(self, uri: str, db: str, data_dir: str, phase: str,
                 limit: Optional[int], max_orderlines: Optional[int],
                 batch_size: int, drop: bool) -> None:
        self.uri = uri
        self.db = db
        self.data_dir = data_dir
        self.phase = phase
        self.limit = limit
        self.max_orderlines = max_orderlines
        self.batch_size = batch_size
        self.drop = drop

    def __repr__(self) -> str:
        return (f"Config(uri={self.uri!r}, db={self.db!r}, data_dir={self.data_dir!r}, "
                f"phase={self.phase!r}, limit={self.limit}, "
                f"max_orderlines={self.max_orderlines}, batch_size={self.batch_size}, "
                f"drop={self.drop})")


def get_config(argv: Optional[list[str]] = None) -> Config:
    """Vrati Config iz NB_* (u notebook-u) ili iz CLI argumenata (kao skripta)."""
    if _in_notebook() and argv is None:
        cfg = Config(NB_URI, NB_DB, NB_DATA_DIR, NB_PHASE, NB_LIMIT, NB_MAX_ORDERLINES,
                     NB_BATCH_SIZE, NB_DROP)
    else:
        a = build_parser().parse_args(argv)
        cfg = Config(a.uri, a.db, a.data_dir, a.phase, a.limit, a.max_orderlines,
                     a.batch_size, a.drop)
    if not os.path.isdir(cfg.data_dir):
        raise FileNotFoundError(f"--data-dir ne postoji: {cfg.data_dir!r}")
    for fname in (PERSON_CSV, PRODUCT_CSV, ORDER_CSV, ORDERLINE_CSV):
        path = os.path.join(cfg.data_dir, fname)
        if not os.path.isfile(path):
            raise FileNotFoundError(f"Nedostaje CSV fajl: {path!r}")
    return cfg

## Pomoćne funkcije: čišćenje vrednosti, kastovanje, streaming čitanje, progres

In [14]:
def clean(value: Optional[str]) -> Optional[str]:
    """Skini whitespace; prazan string ili 'NULL' -> None."""
    if value is None:
        return None
    v = value.strip()
    return None if v in NULL_TOKENS else v


def to_int(value: Optional[str]) -> Optional[int]:
    v = clean(value)
    return None if v is None else int(v)


def to_float(value: Optional[str]) -> Optional[float]:
    v = clean(value)
    return None if v is None else float(v)


def to_datetime(value: Optional[str]) -> Optional[datetime]:
    v = clean(value)
    if v is None:
        return None
    for fmt in DATETIME_FORMATS:
        try:
            return datetime.strptime(v, fmt)
        except ValueError:
            continue
    raise ValueError(f"Neprepoznat format datuma: {v!r}")


def to_id(value: Optional[str]) -> Any:
    """Prirodni ključ -> int kad je moguće, inače očišćen string."""
    v = clean(value)
    if v is None:
        return None
    try:
        return int(v)
    except ValueError:
        return v


def open_rows(path: str) -> Iterator[dict[str, str]]:
    """Stream CSV-a kao dict redovi (po headeru). Ne učitava ceo fajl u memoriju.

    Koristi DictReader pa je otporno na redosled kolona u fajlu (npr. order.csv ima
    total_amount kao poslednju kolonu, ne odmah posle status-a).
    """
    f = open(path, "r", encoding="utf-8", newline="")
    try:
        reader = csv.DictReader(f, delimiter=DELIMITER)
        for row in reader:
            yield row
    finally:
        f.close()


def progress(iterable: Iterable, desc: str) -> Iterable:
    """Vizuelni progres preko tqdm; fallback ispisuje na svakih ~50k redova."""
    if tqdm is not None:
        return tqdm(iterable, desc=desc, unit="row", mininterval=0.5)

    def _gen() -> Iterator:
        for i, item in enumerate(iterable, 1):
            if i % 50_000 == 0:
                print(f"  {desc}: {i:,} redova", flush=True)
            yield item

    return _gen()


def iter_batches(it: Iterator[dict], size: int) -> Iterator[list[dict]]:
    """Grupiši dokumente u batch-eve zadate veličine."""
    batch: list[dict] = []
    for doc in it:
        batch.append(doc)
        if len(batch) >= size:
            yield batch
            batch = []
    if batch:
        yield batch


def insert_stream(coll: Collection, docs: Iterator[dict], batch_size: int, desc: str) -> int:
    """Ubaci dokumente u unordered batch-evima. Vrati broj ubačenih."""
    n = 0
    for batch in iter_batches(progress(docs, desc), batch_size):
        coll.insert_many(batch, ordered=False)
        n += len(batch)
    print(f"  -> {desc}: ubačeno {n:,} dokumenata", flush=True)
    return n


def maybe_drop(db: Database, names: list[str], drop: bool) -> None:
    """Obriši ciljne kolekcije ako je --drop zadato (idempotentnost)."""
    if not drop:
        return
    for name in names:
        db.drop_collection(name)
    print(f"  obrisane kolekcije u {db.name}: {', '.join(names)}", flush=True)


def _limit_order_ids(data_dir: str, limit: Optional[int]) -> Optional[set]:
    """Skup order_id vrednosti prvih N porudžbina (order.csv je sortiran)."""
    if limit is None:
        return None
    ids: set = set()
    for i, row in enumerate(open_rows(os.path.join(data_dir, ORDER_CSV))):
        if i >= limit:
            break
        ids.add(to_id(row["order_id"]))
    return ids


def _line_cutoff(data_dir: str, max_order_id: Optional[int],
                 max_orderlines: Optional[int]) -> Optional[tuple[int, int]]:
    """Tačka preseka za TAČNO `max_orderlines` stavki sa NAJNIŽIM order_id-evima.

    Jedan prolaz kroz orderline.csv prebroji stavke po porudžbini (uz oid <= max_order_id),
    pa kumulativno (rastuće po order_id) nađe presek. Vraća (C, R): zadrži sve stavke
    order_id < C i prvih R stavki order_id == C (po orderline_id). Vraća None ako nema
    ograničenja ili ima manje stavki od ciljanog broja. Pretpostavlja orderline_id 1..n
    po porudžbini (kontigualno) — što važi za ovaj dataset.
    """
    if max_orderlines is None:
        return None
    counts: dict[int, int] = {}
    for row in open_rows(os.path.join(data_dir, ORDERLINE_CSV)):
        oid = int(row["order_id"])
        if max_order_id is not None and oid > max_order_id:
            continue
        counts[oid] = counts.get(oid, 0) + 1
    cum = 0
    for oid in sorted(counts):
        n = counts[oid]
        if cum + n >= max_orderlines:
            return (oid, max_orderlines - cum)
        cum += n
    return None  # manje stavki nego ciljano -> zadrži sve


def _line_kept(oid: int, lid: int, cutoff: Optional[tuple[int, int]]) -> bool:
    """Da li orderline (order_id, orderline_id) ulazi u zadržani skup prema cutoff-u."""
    if cutoff is None:
        return True
    c, r = cutoff
    return oid < c or (oid == c and lid <= r)

## Faza 1 — baseline (`{db}_v1`): ravne string kolekcije, bez indeksa

In [15]:
def phase1_doc(row: dict[str, str]) -> dict:
    """Ravan dokument sa svim vrednostima kao stringovi (prazno/NULL -> None)."""
    return {k: clean(v) for k, v in row.items()}


def load_phase1(client: MongoClient, cfg: Config, order_ids: Optional[set],
                cutoff: Optional[tuple[int, int]]) -> dict[str, int]:
    db = client[f"{cfg.db}_v1"]
    print(f"[Faza 1] baseline -> baza {db.name!r} (ravne string kolekcije, bez indeksa)")
    maybe_drop(db, ["persons", "products", "orders", "orderlines"], cfg.drop)

    counts: dict[str, int] = {}
    p = cfg.data_dir

    # persons / products: uvek u celosti (mali su)
    counts["persons"] = insert_stream(
        db.persons, (phase1_doc(r) for r in open_rows(os.path.join(p, PERSON_CSV))),
        cfg.batch_size, "persons")
    counts["products"] = insert_stream(
        db.products, (phase1_doc(r) for r in open_rows(os.path.join(p, PRODUCT_CSV))),
        cfg.batch_size, "products")

    # orders: prvih N (order_ids); orderlines: tačno `cutoff` stavki (najniži order_id-evi)
    def order_docs() -> Iterator[dict]:
        n = 0
        for row in open_rows(os.path.join(p, ORDER_CSV)):
            if order_ids is not None:
                if to_id(row["order_id"]) not in order_ids:
                    continue
                n += 1
            yield phase1_doc(row)
            if order_ids is not None and n >= len(order_ids):
                break

    counts["orders"] = insert_stream(db.orders, order_docs(), cfg.batch_size, "orders")

    def orderline_docs() -> Iterator[dict]:
        # Baseline je namerno naivan: orderline.csv nije sortiran, pa se ceo prolazi i
        # filtrira po pripadnosti zadržanom skupu (order_ids + prag stavki `cutoff`).
        for row in open_rows(os.path.join(p, ORDERLINE_CSV)):
            oid = int(row["order_id"])
            if order_ids is not None and oid not in order_ids:
                continue
            if not _line_kept(oid, int(row["orderline_id"]), cutoff):
                continue
            yield phase1_doc(row)

    counts["orderlines"] = insert_stream(db.orderlines, orderline_docs(), cfg.batch_size, "orderlines")
    return counts

## Faza 2 — eksterno sortiranje `orderline.csv`
`orderline.csv` nije sortiran po `order_id`. Pre streaming merge-a sortiramo ga eksterno:
(1) podeli na sortirane chunk-ove na disku, (2) lazy k-way merge sortiranih chunk-ova.

In [16]:
def external_sort_orderlines(
    path: str, chunk_rows: int,
    row_filter: Optional[Callable[[dict[str, str]], bool]] = None,
) -> tuple[Iterator[dict[str, str]], Callable[[], None]]:
    """Eksterni merge-sort orderline.csv po (order_id, orderline_id) bez učitavanja u memoriju.

    Vraća (iterator sortiranih redova, cleanup funkciju). Pozovi cleanup() po završetku.
    """
    tmpdir = tempfile.mkdtemp(prefix="orderline_sort_")
    chunk_paths: list[str] = []

    def key(r: dict[str, str]) -> tuple[int, int]:
        return (int(r["order_id"]), int(r["orderline_id"]))

    # 1) podeli u sortirane chunk-ove na disku
    src = open(path, "r", encoding="utf-8", newline="")
    try:
        reader = csv.DictReader(src, delimiter=DELIMITER)
        header = reader.fieldnames
        if header is None:
            raise ValueError(f"Prazan ili neispravan CSV: {path!r}")

        def flush(chunk: list[dict]) -> None:
            chunk.sort(key=key)
            cp = os.path.join(tmpdir, f"chunk_{len(chunk_paths):05d}.csv")
            with open(cp, "w", encoding="utf-8", newline="") as out:
                writer = csv.DictWriter(out, fieldnames=header, delimiter=DELIMITER)
                writer.writeheader()
                writer.writerows(chunk)
            chunk_paths.append(cp)

        chunk: list[dict] = []
        for row in reader:
            if row_filter is not None and not row_filter(row):
                continue
            chunk.append(row)
            if len(chunk) >= chunk_rows:
                flush(chunk)
                chunk = []
        if chunk:
            flush(chunk)
    finally:
        src.close()

    # 2) lazy k-way merge sortiranih chunk-ova (ograničena memorija)
    files = [open(cp, "r", encoding="utf-8", newline="") for cp in chunk_paths]
    readers = [csv.DictReader(f, delimiter=DELIMITER) for f in files]

    def cleanup() -> None:
        for f in files:
            try:
                f.close()
            except OSError:
                pass
        for cp in chunk_paths:
            try:
                os.remove(cp)
            except OSError:
                pass
        try:
            os.rmdir(tmpdir)
        except OSError:
            pass

    merged: Iterator[dict[str, str]] = heapq.merge(*readers, key=key) if readers else iter(())
    print(f"    sortirano u {len(chunk_paths)} chunk-ova (po {chunk_rows:,} redova)", flush=True)
    return merged, cleanup

## Faza 2 — graditelji dokumenata + streaming merge order/orderline

In [17]:
def person_doc(row: dict[str, str]) -> dict:
    return {
        "_id": to_id(row["person_id"]),
        "firstname": clean(row["firstname"]),
        "lastname": clean(row["lastname"]),
        "gender": clean(row["gender"]),
        "age": to_int(row["age"]),
        "address": {
            "street": clean(row["street"]),
            "number": clean(row["streetnumber"]),
            "unit": clean(row["address_unit"]),
            "postalcode": clean(row["postalcode"]),
            "city": clean(row["city"]),
        },
        "phone": clean(row["phone"]),
        "email": clean(row["email"]),
    }


def product_doc(row: dict[str, str]) -> dict:
    return {
        "_id": to_id(row["product_id"]),
        "sku": clean(row["sku"]),
        "name": clean(row["name"]),
        "description": clean(row["description"]),
        "category": clean(row["category"]),
        "subcategory": clean(row["subcategory"]),
        "brand": clean(row["brand"]),
        "price": to_float(row["price"]),
        "cost": to_float(row["cost"]),
        "stock_quantity": to_int(row["stock_quantity"]),
        "dimensions": {
            "weight_kg": to_float(row["weight_kg"]),
            "length_cm": to_float(row["length_cm"]),
            "width_cm": to_float(row["width_cm"]),
            "height_cm": to_float(row["height_cm"]),
        },
        "status": clean(row["status"]),
        "created_date": to_datetime(row["created_date"]),
        "rating": {
            "average": to_float(row["rating_average"]),
            "count": to_int(row["review_count"]),
        },
    }


def line_doc(row: dict[str, str]) -> dict:
    return {
        "line_no": to_int(row["orderline_id"]),
        "product_id": to_id(row["product_id"]),
        "quantity": to_int(row["quantity"]),
        "unit_price": to_float(row["unit_price"]),
        "subtotal": to_float(row["subtotal"]),
        "status": clean(row["status"]),
        "notes": clean(row["notes"]),
    }


def order_doc(row: dict[str, str], lines: list[dict]) -> dict:
    return {
        "_id": to_id(row["order_id"]),
        "person_id": to_id(row["person_id"]),
        "order_date": to_datetime(row["order_date"]),
        "status": clean(row["status"]),
        "total_amount": to_float(row["total_amount"]),
        "payment_method": clean(row["payment_method"]),
        "shipping_method": clean(row["shipping_method"]),
        "notes": clean(row["notes"]),
        "lines": lines,
    }


def merge_orders(
    order_rows: Iterator[dict[str, str]],
    sorted_line_rows: Iterator[dict[str, str]],
) -> Iterator[dict]:
    """Streaming merge: oba ulaza rastuća po order_id; emituj order dokument sa lines[].

    Akumulira stavke tekuće porudžbine i emituje gotov dokument kad se order_id promeni.
    Asertuje da su ulazi sortirani po order_id i prekida sa jasnom porukom ako nisu.
    """
    ol_iter = iter(sorted_line_rows)
    pending = next(ol_iter, None)
    prev_order_id: Optional[int] = None
    last_line_oid: Optional[int] = None

    for orow in order_rows:
        oid = int(orow["order_id"])
        assert prev_order_id is None or oid > prev_order_id, (
            f"order.csv nije sortiran/uniqe po order_id: {oid} posle {prev_order_id}. "
            f"Faza 2 streaming merge zahteva sortiran order.csv."
        )
        prev_order_id = oid

        # preskoči eventualne stavke čiji je order_id ispod tekuće porudžbine (siročad/limit)
        while pending is not None and int(pending["order_id"]) < oid:
            pending = next(ol_iter, None)

        lines: list[dict] = []
        while pending is not None and int(pending["order_id"]) == oid:
            cur_oid = int(pending["order_id"])
            assert last_line_oid is None or cur_oid >= last_line_oid, (
                f"orderline tok nije sortiran po order_id: {cur_oid} posle {last_line_oid}."
            )
            last_line_oid = cur_oid
            lines.append(line_doc(pending))
            pending = next(ol_iter, None)

        yield order_doc(orow, lines)


def _filter_lines_until(rows: Iterator[dict[str, str]], max_order_id: int) -> Iterator[dict[str, str]]:
    """Rani prekid sortiranog toka stavki kad order_id pređe max_order_id (za --limit)."""
    for r in rows:
        if int(r["order_id"]) > max_order_id:
            break
        yield r

## Faza 2 — učitavanje (`{db}_v2`) + indeksi

In [18]:
def create_indexes_phase2(db: Database) -> None:
    """Kreiraj ciljane indekse TEK posle učitavanja (brže)."""
    print("  kreiranje indeksa za Fazu 2 ...", flush=True)
    db.orders.create_index([("order_date", ASCENDING)])
    db.orders.create_index([("person_id", ASCENDING), ("order_date", ASCENDING)])
    db.orders.create_index([("total_amount", DESCENDING)])
    db.orders.create_index([("lines.product_id", ASCENDING)])
    db.products.create_index([("category", ASCENDING)])
    db.products.create_index([("price", ASCENDING)])
    db.persons.create_index([("city", ASCENDING)])
    print("  indeksi kreirani.", flush=True)


def load_phase2(client: MongoClient, cfg: Config, order_ids: Optional[set],
                cutoff: Optional[tuple[int, int]]) -> dict[str, int]:
    db = client[f"{cfg.db}_v2"]
    print(f"[Faza 2] optimizovano -> baza {db.name!r} (tipizirano, ugnježdene stavke)")
    maybe_drop(db, ["persons", "products", "orders"], cfg.drop)

    counts: dict[str, int] = {}
    p = cfg.data_dir

    counts["persons"] = insert_stream(
        db.persons, (person_doc(r) for r in open_rows(os.path.join(p, PERSON_CSV))),
        cfg.batch_size, "persons")
    counts["products"] = insert_stream(
        db.products, (product_doc(r) for r in open_rows(os.path.join(p, PRODUCT_CSV))),
        cfg.batch_size, "products")

    def order_rows() -> Iterator[dict[str, str]]:
        n = 0
        for row in open_rows(os.path.join(p, ORDER_CSV)):
            if order_ids is not None:
                if to_id(row["order_id"]) not in order_ids:
                    continue
                n += 1
            yield row
            if order_ids is not None and n >= len(order_ids):
                break

    # U eksterni sort ulaze SAMO zadržane stavke (order_ids + prag) -> mnogo manji sort.
    row_filter: Optional[Callable[[dict[str, str]], bool]] = None
    if order_ids is not None or cutoff is not None:
        def row_filter(r: dict[str, str]) -> bool:
            oid = int(r["order_id"])
            if order_ids is not None and oid not in order_ids:
                return False
            return _line_kept(oid, int(r["orderline_id"]), cutoff)

    print("  eksterno sortiranje orderline.csv po (order_id, orderline_id) ...", flush=True)
    sorted_lines, cleanup = external_sort_orderlines(
        os.path.join(p, ORDERLINE_CSV), SORT_CHUNK_ROWS, row_filter)
    try:
        if order_ids is not None:
            max_id = max(int(x) for x in order_ids)
            sorted_lines = _filter_lines_until(sorted_lines, max_id)
        order_documents = merge_orders(order_rows(), sorted_lines)
        counts["orders"] = insert_stream(db.orders, order_documents, cfg.batch_size, "orders")
    finally:
        cleanup()

    create_indexes_phase2(db)
    return counts

## Rezime + glavni runner

In [19]:
def summarize(client: MongoClient, cfg: Config) -> None:
    print("\n===== REZIME =====")
    suffixes = []
    if cfg.phase in ("1", "both"):
        suffixes.append("v1")
    if cfg.phase in ("2", "both"):
        suffixes.append("v2")
    for suffix in suffixes:
        db = client[f"{cfg.db}_{suffix}"]
        print(f"Baza {db.name!r}:")
        for name in sorted(db.list_collection_names()):
            print(f"  {name:12s}: {db[name].estimated_document_count():,} dokumenata")


def run(argv: Optional[list[str]] = None) -> dict[str, dict[str, int]]:
    """Učitaj podatke prema konfiguraciji (notebook NB_* ili CLI argv)."""
    cfg = get_config(argv)
    print(cfg)
    client = MongoClient(cfg.uri)
    try:
        client.admin.command("ping")
    except Exception as exc:  # connection/auth greške
        raise SystemExit(f"Ne mogu da se povežem na MongoDB ({cfg.uri}): {exc}") from exc

    # Skup zadržanih porudžbina i prag stavki računamo JEDNOM (deljeno između faza).
    order_ids = _limit_order_ids(cfg.data_dir, cfg.limit)
    max_order_id = max(order_ids) if order_ids else None
    cutoff = _line_cutoff(cfg.data_dir, max_order_id, cfg.max_orderlines)
    if order_ids is not None:
        print(f"  zadržavam prvih {len(order_ids):,} porudžbina (order_id <= {max_order_id}).",
              flush=True)
    if cutoff is not None:
        print(f"  prag stavki: order_id < {cutoff[0]} (sve) ili == {cutoff[0]} sa "
              f"orderline_id <= {cutoff[1]}  =>  ukupno {cfg.max_orderlines:,} stavki.",
              flush=True)

    results: dict[str, dict[str, int]] = {}
    try:
        if cfg.phase in ("1", "both"):
            results["v1"] = load_phase1(client, cfg, order_ids, cutoff)
        if cfg.phase in ("2", "both"):
            results["v2"] = load_phase2(client, cfg, order_ids, cutoff)
        summarize(client, cfg)
    finally:
        client.close()
    print("\nGotovo.")
    return results

## Pokretanje
U notebook-u koristi `NB_*` konstante; kao skripta parsira CLI argumente.

In [20]:
if __name__ == "__main__" or _in_notebook():
    run()

Config(uri='mongodb://localhost:27017', db='ecommerce', data_dir='archive', phase='both', limit=1000000, max_orderlines=1000000, batch_size=10000, drop=True)
  zadržavam prvih 1,000,000 porudžbina (order_id <= 1000000).
  prag stavki: order_id < 384050 (sve) ili == 384050 sa orderline_id <= 3  =>  ukupno 1,000,000 stavki.
[Faza 1] baseline -> baza 'ecommerce_v1' (ravne string kolekcije, bez indeksa)
  obrisane kolekcije u ecommerce_v1: persons, products, orders, orderlines


persons: 600000row [00:13, 45611.38row/s]

  -> persons: ubačeno 600,000 dokumenata



products: 8000row [00:00, 97479.61row/s]


  -> products: ubačeno 8,000 dokumenata


...

  -> products: ubačeno 8,000 dokumenata
  eksterno sortiranje orderline.csv po (order_id, orderline_id) ...


    sortirano u 2 chunk-ova (po 500,000 redova)


orders: 1000000row [00:47, 20968.53row/s]

  -> orders: ubačeno 1,000,000 dokumenata
  kreiranje indeksa za Fazu 2 ...


  indeksi kreirani.

===== REZIME =====
Baza 'ecommerce_v1':
  orderlines  : 1,000,000 dokumenata
  orders      : 1,000,000 dokumenata
  persons     : 600,000 dokumenata
  products    : 8,000 dokumenata
Baza 'ecommerce_v2':
  orders      : 1,000,000 dokumenata
  persons     : 600,000 dokumenata
  products    : 8,000 dokumenata

Gotovo.
